# 05_aspect_declarations — the four declarations, and what they refuse

The machine calls four kinds of method itself — the regular aspect, the summary, the
error handler and the compensator — so their signature is a contract.

Each cell below makes one mistake and prints what the framework says. Every refusal
arrives before the run: at the class body, or at assembly.


In [ ]:
!pip install -q aoa-action-machine

In [2]:
import asyncio
from typing import Any

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.exceptions import (
    ActionResultTypeError,
    AspectSignatureError,
    DuplicateSummaryAspectError,
)
from aoa.action_machine.graph.core.exclude_graph_model import exclude_graph_model
from aoa.action_machine.graph.nodes.action_graph_node import ActionGraphNode
from aoa.action_machine.intents.aspects import regular_aspect, summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult, BaseState, StubParams, StubResult
from aoa.action_machine.resources import BaseResource, Connections
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.action_machine.runtime.tools_box import ToolsBox

In [3]:
class DeclarationsDomain(BaseDomain):
    """Group the declaration cases."""

    name = "declarations"
    description = "What an aspect declaration must be"


def report(case: str, outcome: str) -> None:
    """Print one case and what the framework answered, without the moment sentence."""
    head = outcome.split(" Refused while")[0].strip()
    print(f"{case:<26} {head}")

### The pipeline passes params, state, box, connections. This method says `p` and `s`.

It would run: Python binds by position, so `p` would receive the parameters object.


In [4]:
# expected-refusal: AspectSignatureError — every case below is wrong on purpose
try:

    @summary_aspect("Renamed parameters")
    async def renamed_summary(self, p: StubParams, s: Any, box: Any, connections: Any) -> StubResult:
        return StubResult()

except AspectSignatureError as error:
    report("renamed parameter", str(error))

renamed parameter          @summary_aspect: method 'renamed_summary' must declare its parameters as (self, params, state, box, connections), got (self, p, s, box, connections). The pipeline hands them over in that order and Python binds by position, so a renamed or reordered parameter receives something other than it says.


### Keyword-only parameters cannot receive values the pipeline passes by position.



In [5]:
# expected-refusal: AspectSignatureError
try:

    @regular_aspect("Keyword-only parameters")
    async def keyword_only_aspect(
        self, params: StubParams, state: Any, box: Any, *, connections: Any
    ) -> dict[str, Any]:
        return {}

except AspectSignatureError as error:
    report("keyword-only parameter", str(error))

keyword-only parameter     @regular_aspect: method 'keyword_only_aspect' must be callable with positional arguments; the pipeline passes (self, params, state, box, connections) by position, so keyword-only parameters, *args and **kwargs are refused.


### The annotations are how a step states what it receives; a bare one is refused.



In [6]:
# expected-refusal: AspectSignatureError
try:

    @summary_aspect("No annotations")
    async def untyped_summary(self, params, state, box, connections) -> StubResult:
        return StubResult()

except AspectSignatureError as error:
    report("untyped declaration", str(error))

untyped declaration        @summary_aspect: method 'untyped_summary' must annotate every parameter it receives; (params, state, box, connections) carry no annotation.


### The shape is right here; only the `box` annotation is wrong, and that is visible

only once the operation's own names exist.


In [7]:
# expected-refusal: AspectSignatureError — the shape is right, only the type is wrong
@exclude_graph_model
@meta(description="declarations: a box that is not a tool box", domain=DeclarationsDomain)
@check_roles(GuestRole)
class MistypedAction(BaseAction[StubParams, StubResult]):
    """An operation whose summary promises the wrong type for the tool box."""

    @summary_aspect("Wrong box type")
    async def mistyped_summary(
        self,
        params: StubParams,
        state: BaseState,
        box: str,
        connections: Connections,
    ) -> StubResult:
        """Assemble the result."""
        return StubResult()


try:
    ActionGraphNode(MistypedAction)
except AspectSignatureError as error:
    report("wrong parameter type", str(error))

wrong parameter type       @summary_aspect: MistypedAction.mistyped_summary annotates 'box' as str, but the pipeline hands over (self, params, state, box, connections) in that order: it must be ToolsBox or a subclass of it.


### An operation answers once. The refusal names both methods instead of pointing at a

graph node that was never written.


In [8]:
# expected-refusal: DuplicateSummaryAspectError
@exclude_graph_model
@meta(description="declarations: two exit points", domain=DeclarationsDomain)
@check_roles(GuestRole)
class TwoSummariesAction(BaseAction[StubParams, StubResult]):
    """An operation that declares two summary aspects."""

    @summary_aspect("First exit point")
    async def first_summary(
        self,
        params: StubParams,
        state: BaseState,
        box: ToolsBox,
        connections: Connections,
    ) -> StubResult:
        """Assemble the result, first way."""
        return StubResult()

    @summary_aspect("Second exit point")
    async def second_summary(
        self,
        params: StubParams,
        state: BaseState,
        box: ToolsBox,
        connections: Connections,
    ) -> StubResult:
        """Assemble the result, second way."""
        return StubResult()


try:
    ActionGraphNode(TwoSummariesAction)
except DuplicateSummaryAspectError as error:
    report("two summaries", str(error))

two summaries              TwoSummariesAction declares more than one exit point (first_summary, second_summary); an operation answers once.


### The annotation is honest; the returned value is not. Only the run can see that.



In [9]:
# expected-refusal: ActionResultTypeError
@meta(description="declarations: an answer that is not a Result", domain=DeclarationsDomain)
@check_roles(GuestRole)
class WrongAnswerAction(BaseAction[StubParams, StubResult]):
    """An operation whose summary returns a mapping instead of its Result."""

    @summary_aspect("Answer with a mapping")
    async def wrong_answer_summary(
        self,
        params: StubParams,
        state: BaseState,
        box: ToolsBox,
        connections: Connections,
    ) -> StubResult:
        """Return something that is not the declared Result."""
        return {"not": "a result"}  # type: ignore[return-value]


async def run_wrong_answer() -> None:
    """Run the operation and report what the machine says about its answer."""
    try:
        await ActionProductMachine(cache_coordinator=None).run(Context(), WrongAnswerAction(), StubParams())
    except ActionResultTypeError as error:
        report("answer is not a Result", str(error))

In [10]:
@meta(description="declarations: the contract, met", domain=DeclarationsDomain)
@check_roles(GuestRole)
class CorrectAction(BaseAction[StubParams, StubResult]):
    """An operation whose declaration is the contract."""

    @summary_aspect("Assemble the result")
    async def correct_summary(
        self,
        params: StubParams,
        state: BaseState,
        box: ToolsBox,
        connections: Connections,
    ) -> StubResult:
        """Build the empty result."""
        return StubResult()


async def main() -> None:
    """Run the seventh case; the first six reported as they were declared."""
    answer = await ActionProductMachine(cache_coordinator=None).run(Context(), CorrectAction(), StubParams())
    report("a correct declaration", f"ran, answer: {type(answer).__name__}")
    await run_wrong_answer()


await (main())

a correct declaration      ran, answer: StubResult
answer is not a Result     WrongAnswerAction.wrong_answer_summary returned dict, but the operation declares Result StubResult.
